# Milestone 5 — Experiment F: cross-dataset zero-shot evaluation (transformers)

**Runs on Kaggle or Colab. Inference only — this notebook never trains anything.**

Scores the six Milestone 4 checkpoints (mBERT + XLM-R × seeds {42, 123, 2026})
against the **Notri-Fact holdout**, a corpus none of them has ever seen. That is
RQ2's headline: does a model trained on Ax-to-Grind transfer, or did it learn
Ax-to-Grind's construction?

The classical half of F is already committed and was computed locally in seconds
(A 0.4948, B 0.4787 macro-F1, against in-domain 0.8755 / 0.8835). This notebook
supplies the transformer half, which needs the checkpoints off the HF Hub.

## Why this is a separate notebook from `04_transformer_training.ipynb`

Two differences that are not cosmetic:

1. **It needs a second corpus.** Notebook 04 downloads **Ax-to-Grind only**, and
   says so deliberately — Notri-Fact is the held-out cross-dataset test set and is
   not read until Milestone 5. F evaluates *on* Notri-Fact, so this notebook
   downloads both. Notri-Fact comes from Kaggle and needs its own API token
   (see below), which notebook 04 never required.
2. **It never trains.** There is no `CONFIRM_TRAIN` here and no training cell to
   guard. The expensive cell is guarded by a different flag, `CONFIRM_EVAL` — see
   section 4 for why reusing the training flag would have been actively unsafe.

## Before you start

### On Kaggle
1. **Settings → Accelerator → GPU T4 x2** (or P100). A GPU is strongly preferred
   but *not* required — this is a forward pass; see section 1.
2. **Settings → Internet → On.** Off by default; nothing works without it.
3. **Add-ons → Secrets**, attached to this notebook:
   - `HF_TOKEN` — needs **read** access to pull the checkpoints, and **write**
     access if you want results pushed to the Hub (you do — that is M4-6's
     durability guarantee).
   - `KAGGLE_API_TOKEN` — from kaggle.com/settings/api. Needed to download
     Notri-Fact. **This is new relative to notebook 04.**
4. Set `HF_STAGING_PREFIX` and `REPO_URL` in the restore-state cell below.

### On Colab
Same, but secrets go in the 🔑 sidebar and the accelerator is
`Runtime → Change runtime type`.

## 0. RESTORE STATE — the one cell to re-run after any kernel restart

**If the kernel restarted, died, or you lost your place: run this cell, then carry
on.** It is the only cell you need to re-run.

Idempotent and order-independent — safe to run any number of times, at any point.
It re-establishes `REPO_DIR`, `HF_STAGING_PREFIX`, `os.environ["HF_TOKEN"]`,
`os.environ["KAGGLE_API_TOKEN"]` and `sys.path`, cloning or re-syncing only if
needed.

It also sets `CONFIRM_EVAL = False`. The evaluation cell refuses to run unless you
set that to `True` yourself, so a stray **Run All** cannot start a ~5.4 GB download
and a long scoring job. Restoring state resets it to `False` on purpose — recovering
from a crash must never re-arm expensive work.

In [ ]:
# ---- EDIT THESE, then never touch this cell again ----
HF_STAGING_PREFIX = "your-hf-username"   # e.g. "RehmanAyoub"
REPO_URL = "https://github.com/<your-github-username>/<repo-name>.git"
REPO_BRANCH = "main"
# ------------------------------------------------------

# Guard against an accidental Run All. The evaluation cell refuses to run unless
# YOU set this to True by hand. Reset here on purpose: restoring state must never
# re-arm expensive work.
#
# Deliberately NOT called CONFIRM_TRAIN, and deliberately a different variable —
# see section 4. Nothing in this notebook trains.
CONFIRM_EVAL = False

# Identifies THIS notebook document, for the stale-notebook guard below.
NOTEBOOK_REVISION = 1

import os
import subprocess
import sys

assert HF_STAGING_PREFIX != "your-hf-username", "Set HF_STAGING_PREFIX first."

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle")
REPO_DIR = "/kaggle/working/repo" if ON_KAGGLE else "/content/repo"

# Idempotent by construction: `checkout -B` resets onto origin's tip whether the
# clone is fresh, stale or already current. Untracked files survive a branch reset.
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("cloning...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", REPO_BRANCH], check=True)
subprocess.run(
    ["git", "checkout", "--quiet", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True
)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Credentials, re-read every time rather than "only if unset" — a stale or
# half-set token is exactly the state that fails an hour later at push time.
from research.src.notebook_env import detect_platform, get_secret

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
os.environ["HF_STAGING_PREFIX"] = HF_STAGING_PREFIX

# Notri-Fact lives on Kaggle and needs its own token (research/src/data/download.py
# documents the accepted forms). This is the credential notebook 04 never needed,
# because Milestone 4 only ever touched Ax-to-Grind.
os.environ["KAGGLE_API_TOKEN"] = get_secret("KAGGLE_API_TOKEN")

print("state restored")
print(f"  platform      : {detect_platform()}")
print(f"  REPO_DIR      : {REPO_DIR}  (cwd={os.getcwd()})")
subprocess.run(["git", "log", "--oneline", "-1"], check=True)
print(f"  HF_TOKEN      : {'set' if os.environ.get('HF_TOKEN') else 'MISSING'}")
print(f"  KAGGLE_API_TOKEN: {'set' if os.environ.get('KAGGLE_API_TOKEN') else 'MISSING'}")
print(f"  HF_PREFIX     : {os.environ['HF_STAGING_PREFIX']}")
print(f"  CONFIRM_EVAL  : {CONFIRM_EVAL}  <- the eval cell refuses to run while False")

## 1. Pre-flight — GPU and internet

**A GPU is not mandatory here.** Unlike notebook 04, this is inference: a forward
pass over 13,355 rows per run, six runs. On a T4/P100 that is minutes; on CPU it is
hours but it does produce identical numbers. So a missing GPU is a loud **warning**,
not a hard stop — the `CONFIRM_EVAL` guard is what prevents an accidental long run,
and refusing to start on CPU would block a legitimate (if slow) route.

Internet is still checked hard: Kaggle disables it by default, and without it the
clone, the installs, the checkpoint downloads and the results push all fail.

In [ ]:
import os, socket, subprocess, sys

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle")

# --- GPU: warn, do not exit -------------------------------------------------
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    print(
        "WARNING: no GPU detected.\n"
        "  This notebook is inference-only, so CPU works and gives identical\n"
        "  numbers -- but expect HOURS instead of minutes for 6 runs over\n"
        "  13,355 rows each. To attach one: "
        + ("Settings -> Accelerator -> GPU." if ON_KAGGLE
           else "Runtime -> Change runtime type.")
    )
else:
    print(out.stdout)
    # Pin to one GPU. Kaggle's default is "T4 x2", and multi-GPU changes how the
    # HF Trainer batches (DECISION_REGISTER.md M4-3). Harmless for inference, but
    # kept identical to the training runs so nothing differs between them.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("CUDA_VISIBLE_DEVICES=0 (single GPU, matching the training runs)")

# --- Internet: hard requirement --------------------------------------------
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
    print("internet: OK")
except OSError as exc:
    sys.exit(
        f"No outbound internet ({exc}).\n"
        + ("On Kaggle this is the DEFAULT. Settings -> Internet -> On, then rerun."
           if ON_KAGGLE else "Check the connection and rerun.")
    )

print("platform:", "Kaggle" if ON_KAGGLE else "Colab / other")

## 2. Sync the repo and install pinned dependencies

Identical to notebook 04's section 3 — these cells carry the M4-3 (platform
detection), M4-4 (numpy force-reinstall) and M4-5 (subprocess gate) fixes and are
copied verbatim so the two notebooks cannot drift apart.

The stale-notebook guard compares the notebook you are running against the copy in
the repo. Restarting the runtime reloads the kernel but *not* the notebook source,
so an open tab can otherwise keep executing pre-fix cells against fixed repo code.

In [ ]:
import os, subprocess, sys

# Sync to the branch tip on EVERY run, not just when the directory is missing.
#
# The previous version cloned only if the directory was absent. Restarting the
# session keeps the working directory, so on any rerun the clone was skipped and the
# repo silently stayed at whatever commit it was first cloned at — no output said so.
# Two sources of truth (notebook cells vs repo code) then drift apart with nothing
# reporting it. `checkout -B` resets the local branch onto origin's tip, which is
# idempotent and safe here: research/data/raw/ is gitignored, and untracked files
# (downloaded corpora, produced metrics) are left alone by a hard branch reset.
#
# Plain git throughout. Nothing in this cell is platform-specific beyond REPO_DIR,
# so it behaves identically on Colab and Kaggle — but Kaggle needs internet ENABLED
# for the clone and fetch to work at all (checked in the pre-flight cell).
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", REPO_BRANCH], check=True)
subprocess.run(
    ["git", "checkout", "--quiet", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True
)

# `import research.src...` needs the repo root on sys.path. Colab's IPython puts the
# CWD there implicitly; Kaggle's starts in /kaggle/working and does not.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo synced to:")
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

In [ ]:
# Stale-notebook guard. Compares the notebook YOU are running against the copy just
# synced from the repo, and stops if yours is older.
#
# Why this exists: on 2026-08-15 a run had a correctly-updated repo but a pre-fix
# notebook tab. The cells issued old commands against new code, so the data check ran
# unscoped and the smoke test hit the torchvision error the new cells prevent. Nothing
# reported the mismatch; it looked like the fixes had simply not worked.
import json, re

with open("research/notebooks/05_cross_dataset_eval.ipynb", encoding="utf-8") as fh:
    committed = json.load(fh)

marker = re.compile(r"NOTEBOOK" + r"_REVISION\s*=\s*(\d+)")
found = [
    int(m.group(1))
    for cell in committed["cells"]
    for m in marker.finditer("".join(cell["source"]))
]
repo_revision = max(found) if found else 0

if repo_revision > NOTEBOOK_REVISION:
    raise SystemExit(
        f"STALE NOTEBOOK — you are running revision {NOTEBOOK_REVISION}, the repo has "
        f"revision {repo_revision}.\n\n"
        "Restarting the runtime restarts the kernel; it does NOT reload the notebook "
        "source in an already-open tab.\n"
        "Fix: close this tab, re-open the notebook from GitHub "
        "(File -> Open notebook -> GitHub tab -> this repo), and run from the top."
    )

print(f"notebook revision {NOTEBOOK_REVISION}, repo revision {repo_revision} — in sync")

In [ ]:
# Step 1 of 3 — remove the image's preinstalled torchvision/torchaudio BEFORE
# installing. Applies to Kaggle as much as to Colab: Kaggle's notebook image is
# built FROM the Colab runtime image, so it ships the same preinstalled copies.
#
# They are compiled against whatever torch the image shipped with. The install below
# moves torch to this project's pin, and the leftovers then register C++ ops against
# the wrong ABI, so `from transformers import Trainer` dies with
#     RuntimeError: operator torchvision::nms does not exist
# transformers guards that import with is_torchvision_available(), which only checks
# whether the package is INSTALLED, not whether it imports — a present-but-broken
# copy passes the guard and raises RuntimeError, which nothing on that path catches.
# With torchvision absent the guard is simply False and the block is skipped.
# This project does zero vision and zero audio work. See research/requirements.txt.
#
# pip will warn that fastai/timm now have an unsatisfied torchvision requirement.
# That is expected and harmless — nothing in this pipeline imports them.
!pip uninstall -y -q torchvision torchaudio

# Step 2 of 3 — install the pinned set (REPRODUCIBILITY.md Section 1). On Linux
# (both platforms are Linux) the plain torch pin resolves to the CUDA build,
# which is what a T4 or a P100 needs.
!pip install -q -r research/requirements.txt

# Step 3 of 3 — rewrite numpy and scipy COMPLETELY, over the top of the image's
# copies. Same class of problem as the torchvision block above (a preinstalled
# package left in an inconsistent state), but a different mechanism needing a
# different remedy — and it is NOT a scipy/numpy version conflict, despite the
# traceback looking exactly like one. See DECISION_REGISTER.md M4-4.
#
# What actually failed on Kaggle on 2026-08-19, at `from transformers import Trainer`:
#
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
#                  (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)
#
# That import is INSIDE numpy, not at a scipy/numpy boundary. numpy/_core/umath.py
# is a pure-Python shim that re-exports from the COMPILED _multiarray_umath
# extension, and `_center` is a ufunc living in that binary.
#
# CORRECTED by M4-5: the mixture is in MEMORY, not on disk. The image imports numpy
# at kernel boot; pip then correctly replaces it on disk; the kernel keeps the old
# module objects. The lazy numpy.char/_core.strings then load from the NEW files
# against the OLD cached umath. Both pip and the kernel are right at once — which is
# why the 2026-08-19 run reported numpy 2.0.2 while pip reported 2.5.2 installed.
# The environment gate below therefore runs in a subprocess, where no stale modules
# exist. This force-reinstall is KEPT as cheap insurance against a genuinely partial
# on-disk install, which would look identical from inside the kernel.
#
# scipy is only the messenger. Plain `import numpy` does NOT load the affected
# submodules — numpy.char and numpy._core.strings are lazy. `from numpy import *`
# DOES, and scipy's array_api_compat shim is the first thing in the process to run
# it, which is why every earlier cell, and torch itself, imported fine.
#
# scipy is therefore NOT uninstalled the way torchvision is: it is genuinely
# required. `import sklearn.metrics` — the path every metrics file this project
# writes goes through — pulls in 493 scipy modules, and scikit-learn declares scipy
# as a hard dependency. Removing it would break the run outright.
#
# --force-reinstall rewrites every file of both packages instead of skipping them as
# "already satisfied", which is what repairs the mixed install; --no-deps keeps it
# surgical, so nothing else in the resolved set is disturbed.
!pip install -q --force-reinstall --no-deps numpy==2.5.2 scipy==1.18.0

In [ ]:
# Environment gate — runs in a FRESH SUBPROCESS, not in this kernel. That is the
# fix for DECISION_REGISTER.md M4-5, not a stylistic preference.
#
# Kaggle and Colab import numpy at kernel boot. The dependency cell above then
# replaces numpy on disk, but this kernel keeps the module objects it already
# holds. numpy.char and numpy._core.strings are LAZY, so the first
# `from numpy import *` after the install reads those two files from the NEW numpy
# while numpy._core.umath is still the OLD cached one, and the new strings.py asks
# for a `_center` ufunc the old compiled extension does not have:
#
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
#
# pip and the kernel are BOTH right at the same time, which is why the 2026-08-19
# run reported numpy 2.0.2 (the image's, held in memory) while pip correctly
# reported 2.5.2 installed. Reproduced locally; there is no shadow install and no
# corrupt install on disk.
#
# Every real step below already runs as its own `python -m ...` process, so all of
# them read the freshly installed packages and were never affected. Running the
# gate the same way makes it check the environment the TRAINING actually uses,
# instead of this kernel's stale view of it — and removes any need to restart the
# session mid-notebook.
#
# subprocess.run + an explicit raise, rather than a bare `!python`: a `!` command's
# non-zero exit does NOT stop "Run All", so a failed gate would otherwise scroll by
# and the smoke test would run anyway.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "research/scripts/check_gpu_env.py"],
    cwd=REPO_DIR,
)

if result.returncode != 0:
    raise SystemExit(
        "Environment gate FAILED — see the checks above. Do not continue to the "
        "smoke test; fix the reported items first."
    )

## 3. Data check — **both** corpora

This is the substantive difference from notebook 04, which downloads Ax-to-Grind
only. Experiment F evaluates on the **Notri-Fact holdout**, so both are needed:
Ax-to-Grind because the committed split indices are resolved against it, and
Notri-Fact because it is the target.

Notri-Fact is fetched from Kaggle and needs `KAGGLE_API_TOKEN` (set in section 0).
`research/data/raw/` is gitignored, so this re-downloads on every fresh session.

In [ ]:
# Both datasets — no `--only` filter, unlike notebook 04.
!python -m research.src.data.download

# download.py regenerates MANIFEST.sha256 from whatever is on disk, so verifying
# against it straight after a download would be circular. Restore the COMMITTED
# manifest first — that is the actual dataset-version anchor
# (REPRODUCIBILITY.md Section 3).
!git checkout -- research/data/raw/MANIFEST.sha256

# Full integrity + schema checks, both corpora. If the cross-dataset dedup gate
# were ever to change, this is where it would surface before any scoring happens.
!python -m pytest research/tests/test_raw_data_integrity.py -q
!python -m research.src.data.validate

## 4. Experiment F — score all six checkpoints (GUARDED)

**This is the expensive cell.** Six runs: mBERT and XLM-R × seeds {42, 123, 2026},
each scoring the full 13,355-row Notri-Fact holdout. It downloads ~5.4 GB of
checkpoints and takes minutes on a GPU, hours on CPU.

### Why `CONFIRM_EVAL` and not `CONFIRM_TRAIN`

Notebook 04 guards its training cells with `CONFIRM_TRAIN`. Reusing that flag here
would have been worse than merely inaccurate:

* It is **one global**. Setting `CONFIRM_TRAIN = True` to run an *evaluation* would
  simultaneously arm every training cell in the same kernel — so a subsequent
  "Run All" could start hours of unintended fine-tuning. The guard would be
  actively working against itself.
* It normalises setting the training flag casually. A guard that people learn to
  flip for routine work stops being a guard.

So: same mechanism, distinct flag, independent semantics. Setting one never arms
the other, and each cell names the thing it is actually confirming.

### Durability

Each seed's metrics are pushed to the Hub **as they are computed**, not batched to
the end (`DECISION_REGISTER.md` M4-6 — this is exactly how Milestone 4's results
were lost). A disconnect mid-run therefore costs only the seed in flight; every
finished seed is already safe. That is why there is **no** `--no-push-results` here,
and why `HF_TOKEN` needs write access.

In [ ]:
# EXPENSIVE — guarded. ~5.4 GB of downloads, 6 scoring runs, no training.
#
# To run: put `CONFIRM_EVAL = True` in a cell of your own, then run this one.
# subprocess.run + explicit raise rather than `!python`: a `!` magic's non-zero
# exit does NOT stop Run All, so a failed run would scroll past unnoticed.
import subprocess
import sys

if not globals().get("CONFIRM_EVAL", False):
    print(
        "SKIPPED — CONFIRM_EVAL is not set.\n"
        "This cell scores 6 checkpoints against the Notri-Fact holdout\n"
        "(~5.4 GB of downloads; minutes on GPU, hours on CPU).\n"
        "To run it: put `CONFIRM_EVAL = True` in a cell above, then re-run this.\n"
        "Nothing has been changed."
    )
else:
    result = subprocess.run(
        [sys.executable, "-m", "research.src.experiments.run_cross_dataset",
         "--models", "transformer", "--directions", "F"],
        cwd=REPO_DIR,
    )
    if result.returncode != 0:
        raise SystemExit(
            "Experiment F failed — see the output above. Any seed that finished "
            "before the failure is already on the Hub (M4-6); re-running is safe "
            "and will simply recompute."
        )
    print("\nExperiment F complete. Run the summary and packaging cells below.")

## 5. Summary — the RQ2 comparison, copy this back

In-domain (Milestone 4, committed) beside cross-dataset (this run). The **delta** is
RQ2's headline, and `dominant %` is the one to read closely: a model that has learned
a dataset-specific shortcut typically collapses toward one class on an unseen corpus,
which macro-F1 alone can understate. The classical baselines collapsed to 68.6% and
71.5% — worth comparing against.

In [ ]:
import glob
import json
import statistics
from collections import defaultdict

in_domain, cross = defaultdict(dict), {}

for path in glob.glob("research/results/metrics/*.json"):
    d = json.load(open(path, encoding="utf-8"))
    key = (d["experiment_id"], d["model"], d.get("seed"))
    if d["experiment_id"] in ("C", "D") and d["split"] == "test":
        in_domain[(d["model"], d["seed"])] = d["metrics"]["macro_f1"]
    elif d["experiment_id"] == "F":
        cross[(d["model"], d.get("seed"))] = d

hdr = f"{'model':<32}{'seed':>6}{'in-domain':>11}{'zero-shot':>11}{'delta':>9}{'dominant %':>12}{'collapsed':>11}"
print(hdr); print("-" * len(hdr))
for (model, seed), rec in sorted(cross.items(), key=lambda kv: (str(kv[0][0]), str(kv[0][1]))):
    zs = rec["metrics"]["macro_f1"]
    dom = rec["prediction_collapse"]["dominant_class_share"]
    coll = rec["prediction_collapse"]["is_collapsed"]
    base = in_domain.get((model, seed))
    if base is None:
        print(f"{model:<32}{str(seed):>6}{'n/a':>11}{zs:>11.4f}{'n/a':>9}{dom:>11.2%}{str(coll):>11}")
    else:
        print(f"{model:<32}{str(seed):>6}{base:>11.4f}{zs:>11.4f}{zs-base:>9.4f}{dom:>11.2%}{str(coll):>11}")

print("\nSeed variance of the zero-shot result (EXPERIMENT_PLAN.md Section 5):")
by_model = defaultdict(list)
for (model, seed), rec in cross.items():
    if seed is not None:
        by_model[model].append(rec["metrics"]["macro_f1"])
for model, vals in sorted(by_model.items()):
    if len(vals) > 1:
        print(f"  {model:<32} mean={statistics.mean(vals):.4f} sd={statistics.stdev(vals):.4f} "
              f"values={[round(v, 4) for v in sorted(vals)]}")

## 6. Package — **the run is not done until these are on the Hub**

`DECISION_REGISTER.md` M4-6. Each seed was already pushed as it completed; this is
the sweep that catches anything whose retries failed, verifies against what the Hub
actually lists, and **stops the notebook if it cannot confirm**. Only then does it
build a zip for the Output tab, which is a convenience copy and never the copy.

In [ ]:
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from research.src.evaluation.results_push import (  # noqa: E402
    push_result_files,
    verify_results_uploaded,
)
from research.src.notebook_env import deliver_file, working_root  # noqa: E402

metrics_dir = Path(REPO_DIR) / "research" / "results" / "metrics"
metrics = sorted(metrics_dir.glob("F_*.json"))
assert metrics, "no F_*.json found — did the evaluation cell actually run?"

print("pushing metrics to the Hub...")
print(f"  {push_result_files(metrics, subdir='milestone5/metrics')}")

verification = verify_results_uploaded(
    [p.name for p in metrics], subdir="milestone5/metrics"
)
print(f"  verification: {verification}")

if not verification["verified"]:
    raise SystemExit(
        "RESULTS ARE NOT SAFE YET — "
        f"{len(verification.get('missing', []))} file(s) not on the Hub: "
        f"{verification.get('missing')}\n"
        f"reason: {verification.get('reason', 'see above')}\n\n"
        "Do NOT close this session. Re-run this cell. The metrics exist in\n"
        f"{metrics_dir}, but only there — the exact state M4-6 exists to prevent."
    )

print(f"\nAll {len(metrics)} F metrics files are on the Hub: {verification['repo_id']}")

archive = working_root() / "milestone5_F_metrics.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in metrics:
        zf.write(path, arcname=path.name)
print(f"zipped {len(metrics)} files -> {archive}")
print(deliver_file(archive))